# AI Replace Inpaint Smoke Runner

Standalone Pokecut-style AI Replace flow under inpaint/.

This notebook mirrors the clean SD3.5 runner style: shared setup first, then separate smoke cells and metrics cells for each dataset.

## 1. Install Dependencies

In [ ]:
!pip install -q "diffusers>=0.30.0,<1.0.0" "transformers>=4.40.0" "accelerate>=0.30.0" safetensors ultralytics huggingface_hub opencv-python pillow numpy pandas matplotlib


## 2. Clone Or Update Repo

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/BDT-17/VIN.git"
REPO_DIR = Path("/kaggle/working/VIN")

if REPO_DIR.exists():
    %cd /kaggle/working/VIN
    !git fetch origin main
    !git pull --ff-only origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd /kaggle/working/VIN

PROJECT_DIR = REPO_DIR if (REPO_DIR / "inpaint").exists() else Path.cwd()
print("PROJECT_DIR:", PROJECT_DIR)


## 3. Imports

In [ ]:
import os
import sys
import json
import shutil
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PROJECT_DIR = Path("/kaggle/working/VIN") if Path("/kaggle/working/VIN/inpaint").exists() else Path.cwd()
sys.path = [str(PROJECT_DIR)] + [path for path in sys.path if path != str(PROJECT_DIR)]
%cd {PROJECT_DIR}

from inpaint.config import DEFAULT_CONFIG, AIReplaceConfig
from inpaint.smoke_runner import run_smoke, list_source_images

print("Loaded inpaint flow:", DEFAULT_CONFIG.AI_REPLACE_FLOW)
print("Model:", DEFAULT_CONFIG.MODEL_ID)


## 4. Runtime Check

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu count:", torch.cuda.device_count())
    for idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(idx)
        print(idx, props.name, round(props.total_memory / 1024**3, 2), "GB")


## 5. Hugging Face Login

In [ ]:
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = hf_token or UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    print("No Kaggle HF_TOKEN secret found:", type(exc).__name__)

if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face")
else:
    print("No HF token provided; public models only")


## 6. Dataset Smoke Config

In [ ]:
from pathlib import Path

DATASET_SMOKE_RUNS = {
    "citypersons_bg_yolo": {
        "input_dir": Path("/kaggle/input/datasets/muttahirulislam/citypersons-dataset-with-bg-image/yolo_dir/yolo_dir"),
        "num_images": 3,
    },
    "cityperson_nguyena": {
        "input_dir": Path("/kaggle/input/datasets/nguyenaabcxyzeric/cityperson"),
        "num_images": 3,
    },
    "mot17_02_frcnn": {
        "input_dir": Path("/kaggle/input/datasets/kyoru4444/mot17-02-fcrnn/MOT17-02-FRCNN"),
        "num_images": 3,
    },
    "human_detection_dataset": {
        "input_dir": Path("/kaggle/input/datasets/constantinwerner/human-detection-dataset/human detection dataset/0"),
        "num_images": 3,
    },
}

SMOKE_IMAGES = 3
SEED = 42
DRY_RUN = False
USE_YOLO = True
OUTPUT_BASE = Path('/kaggle/working/ai_replace_smoke')

for name, cfg in DATASET_SMOKE_RUNS.items():
    path = cfg['input_dir']
    count = len(list_source_images(path, limit=5)) if path.exists() else 0
    print(f"{name:24s} exists={path.exists()} sample_count={count} path={path}")


## 7. Smoke Helpers

In [ ]:
def run_inpaint_dataset_smoke(dataset_name, smoke_images=SMOKE_IMAGES):
    cfg = DATASET_SMOKE_RUNS[dataset_name]
    input_dir = cfg['input_dir']
    if not input_dir.exists():
        raise FileNotFoundError(f'{dataset_name} path not found: {input_dir}')
    output_dir = OUTPUT_BASE / dataset_name
    rows, summary = run_smoke(
        input_dir=input_dir,
        output_dir=output_dir,
        num_images=smoke_images,
        seed=SEED,
        load_model=not DRY_RUN,
        use_yolo=USE_YOLO,
    )
    return rows, summary, output_dir


def show_inpaint_metrics(dataset_name):
    output_dir = OUTPUT_BASE / dataset_name
    summary_path = output_dir / 'metrics' / 'metrics_summary.json'
    summary = json.load(open(summary_path, encoding='utf-8'))
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    return summary


def manifest_head(dataset_name, n=20):
    import pandas as pd
    output_dir = OUTPUT_BASE / dataset_name
    df = pd.read_csv(output_dir / 'manifest.csv')
    cols = [
        'accepted', 'reject_reason', 'outside_mask_diff', 'object_mask_inside_ratio',
        'opacity_score', 'background_preservation_score', 'ai_replace_quality_score',
        'source_path', 'output_path',
    ]
    return df[[col for col in cols if col in df.columns]].head(n)


## 8. Optional Dry-Run Wiring Test

In [ ]:
dry_dataset = "citypersons_bg_yolo"
if DATASET_SMOKE_RUNS[dry_dataset]['input_dir'].exists():
    dry_rows, dry_summary = run_smoke(
        input_dir=DATASET_SMOKE_RUNS[dry_dataset]['input_dir'],
        output_dir=Path('/kaggle/working/ai_replace_dry_run') / dry_dataset,
        num_images=1,
        seed=SEED,
        load_model=False,
        use_yolo=False,
    )
    dry_summary
else:
    print('Dry-run dataset missing:', DATASET_SMOKE_RUNS[dry_dataset]['input_dir'])


## 9. CityPersons YOLO Smoke Test

In [ ]:
citypersons_bg_yolo_rows, citypersons_bg_yolo_summary, citypersons_bg_yolo_output = run_inpaint_dataset_smoke(
    "citypersons_bg_yolo",
    smoke_images=SMOKE_IMAGES,
)
citypersons_bg_yolo_output


### CityPersons YOLO Metrics

In [ ]:
citypersons_bg_yolo_metrics = show_inpaint_metrics("citypersons_bg_yolo")
manifest_head("citypersons_bg_yolo")


## 10. CityPerson NguyenA Smoke Test

In [ ]:
cityperson_nguyena_rows, cityperson_nguyena_summary, cityperson_nguyena_output = run_inpaint_dataset_smoke(
    "cityperson_nguyena",
    smoke_images=SMOKE_IMAGES,
)
cityperson_nguyena_output


### CityPerson NguyenA Metrics

In [ ]:
cityperson_nguyena_metrics = show_inpaint_metrics("cityperson_nguyena")
manifest_head("cityperson_nguyena")


## 11. MOT17-02-FRCNN Smoke Test

In [ ]:
mot17_02_frcnn_rows, mot17_02_frcnn_summary, mot17_02_frcnn_output = run_inpaint_dataset_smoke(
    "mot17_02_frcnn",
    smoke_images=SMOKE_IMAGES,
)
mot17_02_frcnn_output


### MOT17-02-FRCNN Metrics

In [ ]:
mot17_02_frcnn_metrics = show_inpaint_metrics("mot17_02_frcnn")
manifest_head("mot17_02_frcnn")


## 12. Human Detection Dataset Smoke Test

In [ ]:
human_detection_rows, human_detection_summary, human_detection_output = run_inpaint_dataset_smoke(
    "human_detection_dataset",
    smoke_images=SMOKE_IMAGES,
)
human_detection_output


### Human Detection Dataset Metrics

In [ ]:
human_detection_dataset_metrics = show_inpaint_metrics("human_detection_dataset")
manifest_head("human_detection_dataset")


## 13. Preview Gallery

In [ ]:
import math
import matplotlib.pyplot as plt
from PIL import Image

PREVIEW_DATASET = "citypersons_bg_yolo"
preview_dir = OUTPUT_BASE / PREVIEW_DATASET / 'previews'
harmonized = sorted(preview_dir.glob('*_harmonized.png'))[:6]
if not harmonized:
    print('No previews found:', preview_dir)
else:
    cols = 3
    rows_n = math.ceil(len(harmonized) / cols)
    plt.figure(figsize=(5 * cols, 5 * rows_n))
    for i, path in enumerate(harmonized, 1):
        plt.subplot(rows_n, cols, i)
        plt.imshow(Image.open(path))
        plt.title(path.name[:48])
        plt.axis('off')
    plt.tight_layout()


## 14. Outside-Mask Diff Preview

In [ ]:
diffs = sorted((OUTPUT_BASE / PREVIEW_DATASET / 'previews').glob('*_diff_outside_mask.png'))[:6]
if not diffs:
    print('No diff previews found')
else:
    cols = 3
    rows_n = math.ceil(len(diffs) / cols)
    plt.figure(figsize=(5 * cols, 5 * rows_n))
    for i, path in enumerate(diffs, 1):
        plt.subplot(rows_n, cols, i)
        plt.imshow(Image.open(path))
        plt.title(path.name[:48])
        plt.axis('off')
    plt.tight_layout()


## 15. Export Outputs

In [ ]:
zip_base = Path('/kaggle/working') / OUTPUT_BASE.name
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=OUTPUT_BASE)
print('Saved export:', zip_path)
